<a href="https://colab.research.google.com/github/madhusowmyanaidu/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/madhusowmyanaidu/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My chosen lane is content refresh prioritization. I want to identify website pages that show signs of declining search performance so that content teams can decide which pages to review first.

I frame this as a classification problem because the target has two possible classes: declining and not declining. The model will use available content and search-performance features to estimate whether a page belongs to the declining class. This prediction will support content review prioritization rather than automatically deciding which pages must be updated.

In [9]:
# Section 1: Check the dataset and its available columns

import pandas as pd
from pathlib import Path

data_path = Path("data/raw/content_refresh_anonymized.csv")

print("Dataset path:", data_path)
print("Dataset exists:", data_path.exists())

Dataset path: data/raw/content_refresh_anonymized.csv
Dataset exists: False


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

My target variable is whether a content page shows a declining search-performance trend.

For the starter dataset, I will use `trend_direction` to define a binary proxy target:

- 1 = the page's observed trend direction is down.
- 0 = the page's observed trend direction is not down.

This is a proxy because it describes an observed trend in the available data rather than proving that the page will decline in the future. The target is derived from `trend_direction`, so I must not use that column as an input feature when predicting the target.

In [10]:
import os

print(os.getcwd())
print(os.listdir())

/content
['.config', 'flyrank-ml-internship-starter', 'sample_data']


In [11]:
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

fatal: destination path 'flyrank-ml-internship-starter' already exists and is not an empty directory.


In [12]:
import pandas as pd

data_path = (
    "/content/flyrank-ml-internship-starter/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(data_path)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)

display(df.head())

Dataset loaded successfully!
Dataset shape: (30000, 44)


,content_id,client_id,search_volume,competition,competition_level,cpc,content_type,main_intent,word_count,char_count,...,char_count_tier,ctr,avg_position,engagement_rate,scroll_rate,ai_traffic_pct,impression_tier,position_tier,trend_direction,trend_pct
0,content_304f48230142,client_f369cb89fc,10.0,0.67,HIGH,2.05,keyword article,transactional,3221.0,20457.0,...,15000-25000,0.76,10.6,5.88,4.55,0.0,good,striking,down,-41.4
1,content_a1fb4e703a9e,client_4e07408562,90.0,0.01,LOW,0.05,keyword article,informational,2481.0,15562.0,...,15000-25000,0.05,20.3,0.00,10.00,0.0,good,page_3_5,down,-57.7
2,content_9aa793d4d895,client_7f2253d7e2,0.0,0.00,LOW,0.00,keyword article,informational,3515.0,23643.0,...,15000-25000,0.09,36.5,0.00,28.57,0.0,good,page_3_5,down,-60.9
3,content_331d6c4de07b,client_19581e27de,10.0,0.00,LOW,0.00,keyword article,commercial,NaN,NaN,...,NaN,0.49,6.2,1.28,3.45,0.0,good,page_1,stable,-13.8
4,content_d99b7a2d90ca,client_3fdba35f04,0.0,0.00,LOW,0.00,keyword article,informational,2803.0,17469.0,...,15000-25000,0.13,44.0,0.00,24.29,0.0,good,page_3_5,down,-34.7


In [13]:
# Section 2: Load the dataset and inspect the target field

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)
print("Available columns:")
print(df.columns.tolist())

if "trend_direction" in df.columns:
    print("\nTrend direction counts:")
    print(df["trend_direction"].value_counts(dropna=False))
else:
    print("\nThe trend_direction column was not found.")

Dataset shape: (30000, 44)
Available columns:
['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']

Trend direction counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*

My primary success metric is Precision@50.

Precision@50 measures the proportion of the 50 highest-priority pages selected by the model that actually belong to the target class.

For example, a Precision@50 of 0.60 means that 30 of the 50 selected pages belong to the target class.

I chose this metric because a content team has limited time and cannot review every page at once. The goal is to help the team identify useful pages to investigate first. I will compare the model's results with a simple baseline and evaluate the predictions on data that was not used for training.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule might prioritize a page if its clicks or impressions fall below a predefined threshold. However, website content performance depends on multiple factors, such as impressions, clicks, click-through rate (CTR), average search position, content freshness, and engagement. A single if-statement cannot easily capture the different combinations and interactions between these factors.

Machine learning can learn patterns from historical data and combine multiple features to classify pages according to their observed performance trends. For example, a page may have declining clicks but increasing impressions, while another may experience a decline in both clicks and impressions. These situations may require different levels of attention.

I will compare the machine learning model against a simple rule-based baseline using the same evaluation metric, Precision@50. If the model identifies more genuinely declining pages among the top 50 selected pages, it may be more useful for prioritizing content reviews.

The model's output will help content editors decide which pages to investigate first for possible updates. However, I will not assume that machine learning is automatically better; its usefulness must be demonstrated through evaluation on held-out data.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.